<a href="https://colab.research.google.com/github/Neo-Ayush-jha/ALBUM/blob/main/Fake_News_Detector_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔍 AI-Based Fake News Detection System
### Using DistilBERT (Lightweight & Fast)
**Project by:** Ayush Kumar, Satinderjit Singh, Rajan Kumar Singh, Maaz Zafar  
**Guide:** Dr. Sartaj Singh | LPU

---
## ✅ What this notebook does:
- Trains a DistilBERT model on a real fake news dataset
- Lets you input **any news text** → Predicts REAL or FAKE
- Lets you input **any news website URL** → Scrapes + Predicts
- Shows **confidence score** with every prediction
- Saves the trained model for reuse

## ⚡ Runtime Setup:
Go to `Runtime → Change runtime type → GPU (T4)` for faster training

---

## STEP 1 — Install Required Libraries

In [1]:
# ⏳ This takes ~2 minutes on first run
!pip install transformers datasets torch scikit-learn newspaper3k requests beautifulsoup4 lxml[html_clean] -q
print("✅ All libraries installed!")

  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 114.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 114.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.1/211.1 kB 12.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.9/105.9 kB 10.5 MB/s eta 0:00:00
✅ All libraries installed!
✅ All

## STEP 2 — Import Libraries

In [2]:
import torch
import pandas as pd
import numpy as np
import requests
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from bs4 import BeautifulSoup

# Check GPU
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️  No GPU found. Go to Runtime → Change runtime type → GPU")

✅ Using device: cuda
   GPU: Tesla T4
✅ Using device: cuda
   GPU: Tesla T4


## STEP 3 — Download Dataset
We use the **WELFake dataset** (~72,000 news articles, publicly available)

In [3]:
# Download WELFake dataset from Kaggle (public dataset)
# Label: 0 = Fake, 1 = Real

# Option A: Download directly (no login needed)
url = "https://raw.githubusercontent.com/nicholasgasior/fake-news-datasets/main/welfake_sample.csv"

try:
    df = pd.read_csv(url)
    print(f"✅ Dataset loaded from GitHub mirror: {len(df)} rows")
except:
    # Fallback: Create a balanced sample dataset if download fails
    print("⚠️  Using built-in sample dataset (internet may be slow)")
    print("   For full dataset: Upload WELFake_Dataset.csv from Kaggle")

    # Built-in sample (for testing the pipeline)
    fake_samples = [
        "BREAKING: Scientists SHOCKED as they discover the moon is made of cheese, NASA confirms",
        "Government putting mind control chemicals in tap water, whistleblower reveals",
        "URGENT: Bill Gates vaccine contains microchip to track all humans globally",
        "SECRET: Aliens have been living among us for decades, leaked documents show",
        "EXPOSED: 5G towers causing coronavirus spread, experts warn citizens to destroy them",
        "SHOCKING: Obama born in Kenya, new birth certificate proves conspiracy",
        "Deep state globalists plan to eliminate 90% of world population by 2025",
        "Doctors don't want you to know this one trick that cures cancer overnight",
        "LEAKED: Election machines programmed to flip votes for establishment candidates",
        "New study proves smoking actually prevents COVID-19, Big Pharma hiding truth",
        "FBI whistleblower reveals Hillary Clinton runs child trafficking ring from pizza shop",
        "Chemtrails confirmed: Planes spraying population control chemicals since 1990",
        "EXCLUSIVE: Fauci admits COVID was created in lab to make billions from vaccines",
        "Flat Earth finally proven by Navy pilot who flew to the edge and photographed the wall",
        "Reptilian politicians exposed: Shape-shifting caught on camera during press conference",
    ] * 20  # Repeat to get more samples

    real_samples = [
        "Federal Reserve raises interest rates by 25 basis points amid inflation concerns",
        "WHO announces new guidelines for COVID-19 vaccine booster dosing schedule",
        "SpaceX successfully launches 60 Starlink satellites into low Earth orbit",
        "Supreme Court rules 6-3 on landmark environmental protection case",
        "UN Climate Summit reaches agreement on carbon emission reduction targets",
        "Apple reports record quarterly earnings driven by iPhone and services revenue",
        "Scientists discover new antibiotic effective against drug-resistant bacteria",
        "Congress passes infrastructure bill allocating funds for roads and broadband",
        "Olympic committee announces 2028 Los Angeles Games schedule and venues",
        "NASA's James Webb Telescope captures deepest infrared image of universe",
        "European Central Bank adjusts monetary policy to address economic slowdown",
        "Study finds Mediterranean diet reduces risk of heart disease by 30 percent",
        "Tech giant Microsoft acquires AI startup for $2.3 billion to expand capabilities",
        "Global temperatures in 2023 were the highest recorded in modern history",
        "FDA approves new Alzheimer's treatment showing promising clinical trial results",
    ] * 20

    df = pd.DataFrame({
        'text': fake_samples + real_samples,
        'label': [0] * len(fake_samples) + [1] * len(real_samples)
    })
    df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"\n📊 Dataset Info:")
print(f"   Total samples: {len(df)}")
print(f"   Columns: {list(df.columns)}")
df.head()

⚠️  Using built-in sample dataset (internet may be slow)
   For full dataset: Upload WELFake_Dataset.csv from Kaggle

📊 Dataset Info:
   Total samples: 600
   Columns: ['text', 'label']


,text,label
0,"SHOCKING: Obama born in Kenya, new birth certi...",0
1,FDA approves new Alzheimer's treatment showing...,1
2,European Central Bank adjusts monetary policy ...,1
3,URGENT: Bill Gates vaccine contains microchip ...,0
4,Government putting mind control chemicals in t...,0


## STEP 4 — Load Full Kaggle Dataset (RECOMMENDED for better accuracy)
**If you have the WELFake dataset from Kaggle, run this cell**

In [12]:
import csv
csv.field_size_limit(2147483647) # Set a very large limit to avoid issues with large text fields

131072

In [14]:
# ============================================================
# OPTION: Upload your own dataset from Kaggle
# Download from: https://www.kaggle.com/datasets/saurabhshahane/fake-news-classification
# Then upload WELFake_Dataset.csv to Colab
# ============================================================

df = pd.read_csv('/content/WELFake_Dataset.csv', engine='python', on_bad_lines='skip')
df = df[['text', 'label']].dropna()
df['label'] = df['label'].astype(int)  # 0=Fake, 1=Real
print(f"✅ Loaded {len(df)} rows from WELFake dataset")
print(df['label'].value_counts())


✅ Loaded 12841 rows from WELFake dataset
label
1    6748
0    6093
Name: count, dtype: int64


## STEP 5 — Preprocessing & Cleaning

In [15]:
def clean_text(text):
    """Clean and normalize news text"""
    if not isinstance(text, str):
        return ""
    text = text.lower()                          # lowercase
    text = re.sub(r'http\S+|www\S+', '', text)   # remove URLs
    text = re.sub(r'<.*?>', '', text)             # remove HTML tags
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'"%-]', ' ', text)  # keep useful chars
    text = re.sub(r'\s+', ' ', text).strip()     # normalize whitespace
    return text[:512]  # Limit to 512 chars (model max)

# Apply cleaning
# Handle both 'text' only or 'title'+'text' column formats
if 'title' in df.columns and 'text' in df.columns:
    df['clean_text'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).apply(clean_text)
elif 'text' in df.columns:
    df['clean_text'] = df['text'].apply(clean_text)
else:
    df.columns = ['text', 'label']  # rename first two cols
    df['clean_text'] = df['text'].apply(clean_text)

# Remove empty rows
df = df[df['clean_text'].str.len() > 20].reset_index(drop=True)

# Use max 8000 samples for speed (increase if you have more GPU time)
MAX_SAMPLES = 8000
if len(df) > MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=42).reset_index(drop=True)
    print(f"📉 Sampled {MAX_SAMPLES} rows for faster training")

print(f"✅ Cleaned dataset: {len(df)} samples")
print(f"   Label distribution: {df['label'].value_counts().to_dict()}")
print(f"\nSample:")
print(f"  Text: {df['clean_text'][0][:100]}...")
print(f"  Label: {'REAL' if df['label'][0]==1 else 'FAKE'}")

📉 Sampled 8000 rows for faster training
✅ Cleaned dataset: 8000 samples
   Label distribution: {1: 4182, 0: 3818}

Sample:
  Text: reuters - myanmar feels sad over a u.s. decision to sanction a military general, a government spokes...
  Label: FAKE


## STEP 6 — Tokenization & Dataset Class

In [16]:
# Load DistilBERT tokenizer (lightweight version of BERT)
print("⏳ Loading DistilBERT tokenizer...")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')
print("✅ Tokenizer loaded!")

MAX_LEN = 128  # Lightweight: 128 tokens (vs 512 for full BERT)

class NewsDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

print(f"✅ Train: {len(X_train)} | Test: {len(X_test)}")

train_dataset = NewsDataset(X_train, y_train, tokenizer, MAX_LEN)
test_dataset = NewsDataset(X_test, y_test, tokenizer, MAX_LEN)

BATCH_SIZE = 16
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE)

print(f"✅ DataLoaders ready (batch size: {BATCH_SIZE})")

⏳ Loading DistilBERT tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

✅ Tokenizer loaded!
✅ Train: 6400 | Test: 1600
✅ DataLoaders ready (batch size: 16)


## STEP 7 — Load DistilBERT Model

In [17]:
print("⏳ Loading DistilBERT model (66MB, much lighter than BERT-base 440MB)...")

model = DistilBertForSequenceClassification.from_pretrained(
    'distilbert-base-uncased',
    num_labels=2  # 0=Fake, 1=Real
)
model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"✅ Model loaded!")
print(f"   Total parameters: {total_params:,} ({total_params/1e6:.1f}M)")
print(f"   Model on: {device}")

⏳ Loading DistilBERT model (66MB, much lighter than BERT-base 440MB)...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


✅ Model loaded!
   Total parameters: 66,955,010 (67.0M)
   Model on: cuda


## STEP 8 — Training

In [18]:
EPOCHS = 3  # 3 epochs is enough for DistilBERT
LEARNING_RATE = 2e-5

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)

total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)

def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    all_preds, all_labels = [], []

    for batch_num, batch in enumerate(data_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
        loss = outputs.loss
        logits = outputs.logits

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.cpu().numpy())

        if (batch_num + 1) % 20 == 0:
            print(f"   Batch {batch_num+1}/{len(data_loader)} | Loss: {loss.item():.4f}")

    acc = accuracy_score(all_labels, all_preds)
    avg_loss = total_loss / len(data_loader)
    return avg_loss, acc

def evaluate(model, data_loader, device):
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    return acc, all_preds, all_labels

# ======================== TRAINING LOOP ========================
print(f"🚀 Starting training for {EPOCHS} epochs...")
print(f"   Training samples: {len(X_train)}")
print(f"   Batches per epoch: {len(train_loader)}")
print("=" * 60)

best_acc = 0
for epoch in range(EPOCHS):
    print(f"\n📌 EPOCH {epoch+1}/{EPOCHS}")
    train_loss, train_acc = train_epoch(model, train_loader, optimizer, scheduler, device)
    val_acc, _, _ = evaluate(model, test_loader, device)

    print(f"   ✅ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"   ✅ Val Accuracy: {val_acc:.4f}")

    if val_acc > best_acc:
        best_acc = val_acc
        torch.save(model.state_dict(), 'best_fake_news_model.pt')
        print(f"   💾 Best model saved! (acc={best_acc:.4f})")

print(f"\n🏆 Training complete! Best accuracy: {best_acc:.4f}")

🚀 Starting training for 3 epochs...
   Training samples: 6400
   Batches per epoch: 400

📌 EPOCH 1/3
   Batch 20/400 | Loss: 0.7102
   Batch 40/400 | Loss: 0.6495
   Batch 60/400 | Loss: 0.5051
   Batch 80/400 | Loss: 0.4497
   Batch 100/400 | Loss: 0.2799
   Batch 120/400 | Loss: 0.2759
   Batch 140/400 | Loss: 0.4988
   Batch 160/400 | Loss: 0.3701
   Batch 180/400 | Loss: 0.1866
   Batch 200/400 | Loss: 0.1675
   Batch 220/400 | Loss: 0.3003
   Batch 240/400 | Loss: 0.1213
   Batch 260/400 | Loss: 0.0994
   Batch 280/400 | Loss: 0.2805
   Batch 300/400 | Loss: 0.0478
   Batch 320/400 | Loss: 0.2096
   Batch 340/400 | Loss: 0.2402
   Batch 360/400 | Loss: 0.1066
   Batch 380/400 | Loss: 0.0384
   Batch 400/400 | Loss: 0.3276
   ✅ Train Loss: 0.3329 | Train Acc: 0.8473
   ✅ Val Accuracy: 0.8756
   💾 Best model saved! (acc=0.8756)

📌 EPOCH 2/3
   Batch 20/400 | Loss: 0.0457
   Batch 40/400 | Loss: 0.0247
   Batch 60/400 | Loss: 0.4426
   Batch 80/400 | Loss: 0.0184
   Batch 100/400 | L

## STEP 9 — Evaluation Report

In [19]:
# Load best model weights
model.load_state_dict(torch.load('best_fake_news_model.pt', map_location=device))

final_acc, final_preds, final_labels = evaluate(model, test_loader, device)

print("=" * 60)
print("📊 FINAL EVALUATION REPORT")
print("=" * 60)
print(f"Overall Accuracy: {final_acc:.4f} ({final_acc*100:.2f}%)")
print()
print(classification_report(
    final_labels, final_preds,
    target_names=['FAKE', 'REAL']
))

📊 FINAL EVALUATION REPORT
Overall Accuracy: 0.9375 (93.75%)

              precision    recall  f1-score   support

        FAKE       0.93      0.94      0.93       764
        REAL       0.94      0.94      0.94       836

    accuracy                           0.94      1600
   macro avg       0.94      0.94      0.94      1600
weighted avg       0.94      0.94      0.94      1600



## STEP 10 — Save Model for Reuse

In [20]:
# Save full model + tokenizer together
model.save_pretrained('./fake_news_detector_model')
tokenizer.save_pretrained('./fake_news_detector_model')

print("✅ Full model saved to: ./fake_news_detector_model/")
print("   You can reload it anytime without retraining!")

# Zip and download (optional)
!zip -r fake_news_model.zip ./fake_news_detector_model/ -q
print("\n📦 Zipped model: fake_news_model.zip")
print("   Go to Files panel → Download it to keep permanently")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Full model saved to: ./fake_news_detector_model/
   You can reload it anytime without retraining!

📦 Zipped model: fake_news_model.zip
   Go to Files panel → Download it to keep permanently


## STEP 11 — Prediction Function (Text OR URL Input)

In [21]:
import torch.nn.functional as F

def scrape_url(url):
    """Scrape text content from a news website URL"""
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
    }
    try:
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Remove scripts and styles
        for tag in soup(['script', 'style', 'nav', 'footer', 'header', 'aside']):
            tag.decompose()

        # Try to get article body
        article = soup.find('article') or soup.find('main') or soup.find('body')
        paragraphs = article.find_all('p') if article else soup.find_all('p')
        text = ' '.join([p.get_text() for p in paragraphs])

        # Also get title
        title = soup.find('title')
        title_text = title.get_text() if title else ''

        full_text = title_text + ' ' + text
        return full_text[:2000] if len(full_text) > 50 else None

    except Exception as e:
        return None


def predict_news(input_text_or_url):
    """
    Predict if news is REAL or FAKE.
    Accepts: plain text OR a news website URL
    """
    model.eval()

    # Detect if input is a URL
    is_url = input_text_or_url.strip().startswith(('http://', 'https://', 'www.'))

    if is_url:
        print(f"🌐 URL detected. Scraping: {input_text_or_url}")
        text = scrape_url(input_text_or_url)
        if not text:
            print("❌ Could not scrape content from this URL.")
            print("   Try: (1) Check internet connection, (2) Paste the article text directly")
            return
        print(f"✅ Scraped {len(text)} characters\n")
        print(f"📄 Content preview: {text[:200]}...\n")
    else:
        text = input_text_or_url

    # Clean + Tokenize
    clean = clean_text(text)

    encoding = tokenizer(
        clean,
        max_length=MAX_LEN,
        padding='max_length',
        truncation=True,
        return_tensors='pt'
    )

    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)

    with torch.no_grad():
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = F.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred].item() * 100

    fake_prob = probs[0][0].item() * 100
    real_prob = probs[0][1].item() * 100

    label = "✅ REAL NEWS" if pred == 1 else "🚨 FAKE NEWS"

    print("=" * 55)
    print(f"  RESULT: {label}")
    print("=" * 55)
    print(f"  Confidence : {confidence:.1f}%")
    print(f"  REAL Score : {real_prob:.1f}%")
    print(f"  FAKE Score : {fake_prob:.1f}%")

    if confidence < 60:
        print(f"  ⚠️  LOW CONFIDENCE — Verify manually")
    print("=" * 55)

    return {
        'prediction': 'REAL' if pred == 1 else 'FAKE',
        'confidence': round(confidence, 2),
        'real_prob': round(real_prob, 2),
        'fake_prob': round(fake_prob, 2)
    }

print("✅ predict_news() function is ready!")
print("   Usage: predict_news('your text here')")
print("   Usage: predict_news('https://news-website.com/article')")

✅ predict_news() function is ready!
   Usage: predict_news('your text here')
   Usage: predict_news('https://news-website.com/article')


## STEP 12 — 🧪 Test with Text Input

In [22]:
# ============================================================
# TEST 1: FAKE NEWS EXAMPLE
# ============================================================
fake_text = """
BREAKING: Scientists at NASA have confirmed that the moon is actually hollow
and contains an alien civilization that has been watching Earth for centuries.
The government has been hiding this secret for over 50 years. Whistleblowers
from Area 51 have leaked classified documents proving this shocking discovery.
The mainstream media refuses to report this because they are controlled by
globalist elites. Share this before it gets deleted!
"""

print("TEST 1 — FAKE NEWS SAMPLE:")
result1 = predict_news(fake_text)

TEST 1 — FAKE NEWS SAMPLE:
  RESULT: ✅ REAL NEWS
  Confidence : 99.7%
  REAL Score : 99.7%
  FAKE Score : 0.3%


In [23]:
# ============================================================
# TEST 2: REAL NEWS EXAMPLE
# ============================================================
real_text = """
The Federal Reserve raised its benchmark interest rate by 25 basis points on Wednesday,
bringing it to a range of 5.25% to 5.50%, the highest level in 22 years. The decision
was unanimous among the Federal Open Market Committee members. Fed Chair Jerome Powell
stated that the central bank remains committed to bringing inflation back to its 2%
target while carefully monitoring the impact on the labor market and broader economy.
"""

print("TEST 2 — REAL NEWS SAMPLE:")
result2 = predict_news(real_text)

TEST 2 — REAL NEWS SAMPLE:
  RESULT: 🚨 FAKE NEWS
  Confidence : 99.8%
  REAL Score : 0.2%
  FAKE Score : 99.8%


## STEP 13 — 🌐 Test with URL Input

In [24]:
# ============================================================
# TEST 3: URL INPUT — paste any news article URL here
# ============================================================

# Example URLs (replace with any news URL you want to check)
url_to_check = "https://www.bbc.com/news"  # Replace with any news URL

print("TEST 3 — URL INPUT:")
result3 = predict_news(url_to_check)

TEST 3 — URL INPUT:
🌐 URL detected. Scraping: https://www.bbc.com/news
✅ Scraped 2000 characters

📄 Content preview: BBC News - Breaking news, video and the latest top stories from the U.S. and around the world President Trump says “very good conversations” are happening with Tehran but the US won’t be “blackmailed”...

  RESULT: 🚨 FAKE NEWS
  Confidence : 97.6%
  REAL Score : 2.4%
  FAKE Score : 97.6%


## STEP 14 — 🎯 Interactive Checker (Your Custom Input)

In [25]:
# ============================================================
# INTERACTIVE: Enter your own text or URL below
# ============================================================

print("📰 FAKE NEWS DETECTOR — Interactive Mode")
print("Enter news text OR a URL to check:")
print()

# ✏️ REPLACE the text below with your own news or paste a URL
user_input = """
PASTE YOUR NEWS TEXT OR URL HERE
"""

if "PASTE YOUR" in user_input:
    print("⚠️  Please replace the placeholder text above with real news content or a URL")
else:
    result = predict_news(user_input.strip())
    print("\nFull result dict:", result)

📰 FAKE NEWS DETECTOR — Interactive Mode
Enter news text OR a URL to check:

⚠️  Please replace the placeholder text above with real news content or a URL


## STEP 15 — (Optional) Reload Model Without Retraining

In [26]:
# ============================================================
# Run this cell ONLY if you want to reload a saved model
# without retraining (e.g., in a new session)
# ============================================================

# from transformers import DistilBertForSequenceClassification, DistilBertTokenizerFast
# import torch

# model = DistilBertForSequenceClassification.from_pretrained('./fake_news_detector_model')
# tokenizer = DistilBertTokenizerFast.from_pretrained('./fake_news_detector_model')
# device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
# model = model.to(device)
# MAX_LEN = 128
# print("✅ Model reloaded from saved checkpoint!")

print("ℹ️  Uncomment lines above if you need to reload a previously saved model")

ℹ️  Uncomment lines above if you need to reload a previously saved model


---
## 📌 Quick Reference

| What you want | What to do |
|---|---|
| Check news text | `predict_news("paste your text here")` |
| Check a news URL | `predict_news("https://...url...")` |
| Reload saved model | Uncomment Step 15 |
| Download model | Files panel → right-click `fake_news_model.zip` |
| Better accuracy | Upload WELFake_Dataset.csv from Kaggle (Step 4) |

## ⚠️ Important Notes
- Model trained on English news only
- URL scraping may not work on paywalled sites
- Confidence < 60% means borderline — verify manually
- For best accuracy, train on full WELFake dataset (72k rows)